# 04 - SHAP analysis

Attribution over the frozen production model.

> **These notebooks define no functions.** Everything they call lives in `src/`.
> That rule is from `brain.md` section 7: logic written in a cell cannot be tested
> and silently drifts from the module, which is how report figures stop matching
> the code that ships.
>
> Until real case data is in `data/raw/cases/`, these fall back to a generated
> panel and every number describes a generator rather than dengue.

> Requires a frozen artifact. Run `python scripts/freeze_production.py --synthetic` first.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config import load_config

cfg = load_config("../config.yaml")
cfg.project.name, cfg.project.granularity

In [ ]:
from src.panel import assemble_panel
from src.preprocess import preprocess

try:
    panel = assemble_panel(cfg)
    SYNTHETIC = False
except Exception as error:
    print("no real data yet, using the synthetic stand-in:")
    print(" ", str(error).splitlines()[0])
    from src.synthetic import synthetic_panel
    panel = synthetic_panel(cfg)
    SYNTHETIC = True

clean = preprocess(panel, cfg).panel[list(panel.columns)]
clean.shape

## Load the frozen model

The artifact, never a freshly trained model: the explanation must describe the model that makes the forecasts.

In [ ]:
from src.production import load_production

model = load_production()
print(f'{model.experiment}  horizon {model.spec.horizon}  {model.spec.n_features} features')
print(f'trained {model.trained_at.date()}')

## Attribute

KernelExplainer over a flat wrapper. DeepExplainer does not support TF 2.x recurrent layers and has not for years.

In [ ]:
from src.explain import explain
from src.features import build_features

X, y, spec = build_features(clean, model.cfg, horizon=model.spec.horizon)
scaled = (X - model.scaler_mean) / model.scaler_scale
attribution = explain(model.predictor, scaled, spec, cfg)
attribution.values.shape

## What drives the model

Read in domain terms, never as feature indices.

In [ ]:
attribution.global_importance(spec).head(12)[['readable', 'mean_abs_shap']]

## Summed back to raw drivers

The view that goes in the report: 'rainfall matters' is actionable, 'rainfall_roll3_mean at t-7' is not.

In [ ]:
attribution.by_raw_variable(spec)

## One prediction explained

This is what the recommendation layer quotes.

In [ ]:
state, date = attribution.sample_index[0]
print(f'{state}, forecast origin {date.date()}')
for label, value in attribution.top_drivers(spec, row=0, k=5):
    print(f'  {label:<40} {value:+.5f}')

## Per-state rankings

Reported as a finding, not used for selection: a pooled model has one input matrix, so it cannot act on per-state feature sets.

In [ ]:
from src.explain import per_state_ranking

ranking = per_state_ranking(attribution, spec)
{state: ranking[state].idxmax()[1] for state in ranking.columns}

## Feature selection

The top-k feeds back into `features.selected_columns` as an ablation variant.

In [ ]:
from src.explain import select_features

select_features(attribution, spec, cfg)